In [1]:
from langchain.agents import create_agent
from langraph_prompts import ACTION_PROMPT_TEMPLATE
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt
from datetime import datetime

In [2]:
@tool
def get_time_off_balance(user_id: str) -> int:
    '''
    Get the time off balance for a user. 

    Args :
        user_id: A unique string representing the user id

    Returns:
        An integer value representing the time-off balance
    '''
    return 10

@tool
def process_time_off(user_id: str, start_date: str, end_date: str) -> dict:
    '''
    Process time-off request for a user

    Args :
        user_id: A unique string representing the user id
        start_date: A date string in YYYY-MM-DD format for the start date of the time-off request
        end_date: A date string in YYYY-MM-DD format for the end date of the time-off request

    Returns:
        A dictionary with the following keys:
        - status: A string value representing the status of the time-off request
        - message: A string value representing the message of the time-off request
    '''
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    end_date = datetime.strptime(end_date, "%Y-%m-%d")

    days_requested = end_date - start_date
    time_off_balance = get_time_off_balance.invoke({"user_id": user_id})

    result = {}

    if days_requested.days + 1 > time_off_balance:
        result['status'] = 'error'
        result['message'] = f'Time off request failed. you only have {time_off_balance} days of remaining leaves but requested {days_requested.days} days'

    else:
        time_off_balance -= days_requested.days + 1
        result['status'] = 'success'
        result['message'] = f'Time off request processed successfully for {days_requested.days + 1} days. Your remaining time off balance is {time_off_balance}'


    return result


@tool
def get_additional_info_from_human( message: str) -> str:
    '''
        Raises an interrupt to fetch additional information from the human requesting the action

        Args:
            message: a message string from the AI requesting the user for missing information

        Returns:
            A string value representing the response with additional information from the human

    '''
    interrupt_result = interrupt(message)

    return interrupt_result['user_message']


In [3]:
user_id = '1' 
llm = init_chat_model(model="gpt-4o", temperature=0)
tools = [get_time_off_balance, process_time_off, get_additional_info_from_human]

todays_date = datetime.now().strftime("%Y-%m-%d")

system_prompt = PromptTemplate(template=ACTION_PROMPT_TEMPLATE).invoke({'todays_date': todays_date, 'user_id': user_id})

agent = create_agent(llm, tools=tools, system_prompt = system_prompt.text, checkpointer=InMemorySaver())

In [4]:
user_message = "I want to take leave"

config = {'configurable': {'thread_id': user_id}}

In [5]:
result = agent.invoke({"messages": [{"role": "user", "content": user_message}]}, config=config)

result

{'messages': [HumanMessage(content='I want to take leave', additional_kwargs={}, response_metadata={}, id='70ad1be7-688f-4736-afdd-a5569c348b05'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 343, 'total_tokens': 373, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_689bad8e9a', 'id': 'chatcmpl-Cih8ouCKf1O1oAPEjcnud1ENoicik', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--56b89def-9c7b-46f3-8f44-29bedaf00ee9-0', tool_calls=[{'name': 'get_additional_info_from_human', 'args': {'message': 'Please provide the start and end dates for your leave request.'}, 'id': 'call_vtP1zW0cj4iT6IyEefB7kIX0', 'type': 'tool_call'}],

In [6]:
user_message = "tomorrow"

result = agent.invoke(Command(resume={'user_message': user_message}), config=config)

result

{'messages': [HumanMessage(content='I want to take leave', additional_kwargs={}, response_metadata={}, id='70ad1be7-688f-4736-afdd-a5569c348b05'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 343, 'total_tokens': 373, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_689bad8e9a', 'id': 'chatcmpl-Cih8ouCKf1O1oAPEjcnud1ENoicik', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--56b89def-9c7b-46f3-8f44-29bedaf00ee9-0', tool_calls=[{'name': 'get_additional_info_from_human', 'args': {'message': 'Please provide the start and end dates for your leave request.'}, 'id': 'call_vtP1zW0cj4iT6IyEefB7kIX0', 'type': 'tool_call'}],

## Middleware

In [7]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

In [8]:
agent_with_middleware = create_agent(llm, 
    tools=tools, 
    system_prompt = system_prompt.text, 
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "process_time_off": {
                    "allowed_decisions": ["approve", "reject"],
                },
            }
        ),
    ],
    )

In [9]:
user_message = "I want to take leave on 07-12-2025"
config = {'configurable': {'thread_id': user_id}}

In [10]:
agent_with_middleware.invoke({"messages": [{"role": "user", "content": user_message}]}, config=config)

{'messages': [HumanMessage(content='I want to take leave on 07-12-2025', additional_kwargs={}, response_metadata={}, id='4d530373-6fad-493b-8088-e9eddf0a627e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 351, 'total_tokens': 368, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_689bad8e9a', 'id': 'chatcmpl-Cih9IQd1DBMlxQTHpdhWD5LFn3SZ9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--e46a166b-4d03-48e3-b6b6-cedb78d9a870-0', tool_calls=[{'name': 'get_time_off_balance', 'args': {'user_id': '1'}, 'id': 'call_mPIM9jAYix3EyEnhjSeFsZkW', 'type': 'tool_call'}], usage_metadata={'input_tokens': 351, 'output_tokens': 17

In [11]:
agent_with_middleware.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)

{'messages': [HumanMessage(content='I want to take leave on 07-12-2025', additional_kwargs={}, response_metadata={}, id='4d530373-6fad-493b-8088-e9eddf0a627e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 351, 'total_tokens': 368, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_689bad8e9a', 'id': 'chatcmpl-Cih9IQd1DBMlxQTHpdhWD5LFn3SZ9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--e46a166b-4d03-48e3-b6b6-cedb78d9a870-0', tool_calls=[{'name': 'get_time_off_balance', 'args': {'user_id': '1'}, 'id': 'call_mPIM9jAYix3EyEnhjSeFsZkW', 'type': 'tool_call'}], usage_metadata={'input_tokens': 351, 'output_tokens': 17